# Explore ESCI: from search queries to SFT examples

**Question:** does this product satisfy what the shopper searched for?

ESCI provides query–product pairs with four relevance labels. **Supervised fine-tuning (SFT)** turns those pairs into conversations: instructions → query/products → correct answer.

- **Exact:** matches the request.
- **Substitute:** serves the same purpose but differs from what was requested.
- **Complement:** an accessory or companion used with the requested product.
- **Irrelevant:** does not satisfy the request or provide a useful substitute/complement.

Run top to bottom using **this folder's `.venv/bin/python`**. This short notebook reads local files, shows real examples, and calculates descriptive statistics. No training, model downloads, or API calls. All analysis code is visible here. Saved outputs let you browse immediately.

In [ ]:
from pathlib import Path
from collections import Counter
from html import escape
import hashlib
import json

import pandas as pd
import pyarrow.parquet as pq
import matplotlib.pyplot as plt
from IPython.display import display, HTML

# Work from either the project folder or its parent (e.g. Documents in VS Code).
ROOT = next(
    (candidate for base in [Path.cwd(), *Path.cwd().parents]
     for candidate in (base, base / "esci-sft", base / "labs/esci-sft")
     if (candidate / "scripts/prepare_esci_sft.py").is_file()),
    None,
)
assert ROOT is not None, "Open this notebook from the esci-sft lab or AI Buildcraft checkout."
DATA = ROOT / "data/sft"
assert (DATA / "train.jsonl").is_file(), "First run scripts/download.sh and scripts/prepare_esci_sft.py --small; generated data is not bundled."
LABELS = ["Exact", "Substitute", "Complement", "Irrelevant"]
pd.set_option("display.max_colwidth", 100)
pd.set_option("display.max_columns", 15)
print(f"Reading: {DATA}")

## 1. Load the viewing copy and the evaluation splits

`train.jsonl` has one JSON object per line. An optional `train.json` may contain the same objects inside one indented JSON array. When present, we check that it matches exactly; JSONL remains the source of truth.

Every record has **messages** (system instructions, user input, gold assistant answer) and **meta** (IDs and labels for tracing/evaluation). Metadata must stay outside model prompts. When evaluating a model, remove the gold assistant turn too.

In [ ]:
FILES = {
    "train": ["train.jsonl"],
    "val": ["val.jsonl"],
    "test": ["test_pointwise.jsonl", "test_listwise.jsonl"],
}
records_by_split = {}
rows = []
for split, names in FILES.items():
    records_by_split[split] = []
    for name in names:
        for line_number, line in enumerate((DATA / name).read_text().splitlines(), 1):
            record = json.loads(line)
            index = len(records_by_split[split])
            records_by_split[split].append(record)
            meta = record["meta"]
            rows.append({
                "split": split, "record_index": index, "format": meta["format"],
                "query_id": str(meta["query_id"]), "query": meta["query"],
                "product_count": len(meta["product_ids"]), "labels": meta["labels"],
                "product_ids": meta["product_ids"], "example_ids": meta["example_ids"],
                "source_file": name, "source_line": line_number,
            })

viewing_path = DATA / "train.json"
viewing_copy = json.loads(viewing_path.read_text()) if viewing_path.exists() else records_by_split["train"]
assert viewing_copy == records_by_split["train"], (
    "The viewing copy is stale. Re-export train.json from the current train.jsonl."
)
stats = json.loads((DATA / "stats.json").read_text())
# Saved token statistics apply only to the exact files measured by the builder.
for name, info in stats["output_files"].items():
    assert hashlib.sha256((DATA / name).read_bytes()).hexdigest() == info["sha256"], name
examples = pd.DataFrame(rows)
items = examples.explode(["labels", "product_ids", "example_ids"], ignore_index=True)
items = items.rename(columns={"labels": "label", "product_ids": "product_id", "example_ids": "example_id"})
print(f"Loaded {len(viewing_copy):,} training records. Optional viewing copy verified: {viewing_path.exists()}")

## 2. How much data is here?

A **record** is one conversation. A pointwise record makes one product decision; a listwise record makes 4–8 decisions. Count these separately. The downloaded Parquet files are much larger than the prepared sample; reading their metadata below does not load all descriptions into memory.

In [ ]:
raw_files = []
for path in sorted((ROOT / "data/raw").glob("*.parquet")):
    parquet = pq.ParquetFile(path)
    raw_files.append({"file": path.name, "rows": parquet.metadata.num_rows,
                      "size_MB": round(path.stat().st_size / 1_000_000, 1),
                      "columns": ", ".join(parquet.schema_arrow.names)})
display(pd.DataFrame(raw_files))

summary = examples.groupby(["split", "format"], sort=False).agg(
    records=("query_id", "size"), unique_queries=("query_id", "nunique"),
    product_decisions=("product_count", "sum"),
)
display(summary)
train = examples[examples["split"].eq("train")]
print(f"Training: {len(train):,} conversations containing {train.product_count.sum():,} product decisions.")
print("Unique queries per format can overlap within a split; do not add those counts together.")

## 3. Which labels appear, and how often?

The table counts **product decisions**, so each listwise conversation contributes several labels. Training pointwise targets 40% Exact, 30% Substitute, 20% Irrelevant, and 10% Complement. Validation/test pointwise use query-based sampling without label quotas; small samples can vary substantially.

In [ ]:
label_counts = pd.crosstab([items["split"], items["format"]], items["label"])
label_counts = label_counts.reindex(index=summary.index, columns=LABELS, fill_value=0)
display(label_counts)
display((label_counts.div(label_counts.sum(axis=1), axis=0) * 100).round(1).rename_axis(columns="label share (%)"))

fig, axes = plt.subplots(1, 2, figsize=(11, 3.5))
counts = label_counts.loc[("train", "pointwise")]
bars = axes[0].bar(counts.index, counts.values, color="#426b8e")
axes[0].bar_label(bars, padding=3)
axes[0].set(title="Training pointwise labels", xlabel="ESCI label", ylabel="Records")
axes[0].set_ylim(0, counts.max() * 1.18)
sizes = train.loc[train["format"].eq("listwise"), "product_count"].value_counts().reindex(range(4, 9), fill_value=0)
bars = axes[1].bar(sizes.index, sizes.values, color="#426b8e")
axes[1].bar_label(bars, padding=3)
axes[1].set(title="Training list sizes", xlabel="Products per conversation", ylabel="Records", xticks=range(4, 9))
axes[1].set_ylim(0, max(1, sizes.max()) * 1.18)
fig.suptitle("Source: current local train.jsonl")
fig.tight_layout()
plt.show()

## 4. Read actual conversations

The helper below simply displays a saved record with real line breaks. It does not generate or rewrite answers. Set `show_system=True` to see the exact rubric. The metadata table connects each output label to an original product and example ID.

In [ ]:
def show_example(split, record_index, show_system=False):
    record = records_by_split[split][record_index]
    meta = record["meta"]
    display(HTML(f"<h3>{escape(split)} / {escape(meta['format'])} / index {record_index}</h3>"))
    for message in record["messages"]:
        if message["role"] == "system" and not show_system:
            continue
        content = message["content"]
        if message["role"] == "assistant" and meta["format"] == "listwise":
            content = json.dumps(json.loads(content), indent=2, ensure_ascii=False)
        # Escape source text so product HTML or Markdown cannot alter the display.
        display(HTML(
            f"<strong>{escape(message['role'].upper())}</strong>"
            f"<pre style='white-space:pre-wrap;overflow-wrap:anywhere'>{escape(content)}</pre>"
        ))
    display(pd.DataFrame({
        "position": range(1, len(meta["labels"]) + 1),
        "product_id": meta["product_ids"], "source_example_id": meta["example_ids"],
        "gold_label": meta["labels"],
    }))
    print(f"Query ID: {meta['query_id']}. Labels above are source annotations, not model predictions.")

In [ ]:
# A real pointwise example: one product and one answer.
pointwise = train[train["format"].eq("pointwise")]
book = pointwise[pointwise["query"].eq("books for toddlers")]
point_index = int((book if not book.empty else pointwise).iloc[0]["record_index"])
show_example("train", point_index, show_system=True)

In [ ]:
# A real listwise example: one query, several products, one JSON answer.
# This format labels products together; it does not ask for a sorted ranking.
listwise = train[train["format"].eq("listwise")]
cases = listwise[listwise["query"].eq("otterbox case for samsung a20")]
list_index = int((cases if not cases.empty else listwise).iloc[0]["record_index"])
show_example("train", list_index)

## 5. Find your own examples

Edit the four filters, rerun the cell, and choose a `record_index`. Use `None` for all formats or labels and `""` for all queries. For listwise records, a label filter means **at least one product** has that label. Search is case-insensitive literal text, not a regular expression.

In [ ]:
SPLIT = "train"                 # "train", "val", or "test"
FORMAT = "pointwise"            # "pointwise", "listwise", or None
LABEL = "Complement"            # any label from LABELS, or None
QUERY_CONTAINS = "airpods"      # try "watch", "shoes", "mug", or ""

mask = examples["split"].eq(SPLIT)
if FORMAT is not None:
    mask &= examples["format"].eq(FORMAT)
if LABEL is not None:
    mask &= examples["labels"].map(lambda values: LABEL in values)
mask &= examples["query"].str.contains(QUERY_CONTAINS, case=False, regex=False)
matches = examples.loc[mask]
print(f"{len(matches):,} matching conversations; showing at most 20.")
display(matches[["record_index", "format", "query", "labels", "product_count", "source_file", "source_line"]].head(20))

In [ ]:
# Replace None with any record_index from the table, then rerun this cell.
RECORD_INDEX = None
if RECORD_INDEX is not None:
    show_example(SPLIT, RECORD_INDEX)
elif not matches.empty:
    show_example(SPLIT, int(matches.iloc[0]["record_index"]))
else:
    print("No matches. Broaden the filters above and rerun the search.")

## 6. Token lengths and build filtering

These token statistics were measured by the builder with the model's chat template, including system, user, and gold assistant messages. **Characters and words are not tokens.** We read the saved measurements rather than load a tokenizer again. p50 is the median; p95 is the length at or below which about 95% of accepted records fall. Length drops count rejected candidates; the builder sampled replacements.

In [ ]:
length_rows = []
for split, formats in stats["token_lengths_by_split_format"].items():
    for format_name, values in formats.items():
        audit = stats["requested_and_actual"][split][format_name]
        length_rows.append({"split": split, "format": format_name, **values,
                            "limit": stats["tokenizer"][format_name + "_max_tokens"],
                            "length_drops": audit["length_filtered"], "shortfall": audit["shortfall"]})
print("Tokenizer used by builder:", stats["tokenizer"]["model"])
display(pd.DataFrame(length_rows).set_index(["split", "format"]).round(2))
display(pd.DataFrame([stats["source_audit"]]).T.rename(columns={0: "source_count"}))

## 7. Check the boundaries yourself

Training, validation, and test must contain different query IDs. Each original example ID must be used once across all formats. These checks concern dataset structure, not model accuracy. No model has been trained or evaluated by this notebook.

In [ ]:
query_sets = {split: set(examples.loc[examples["split"].eq(split), "query_id"]) for split in FILES}
for left, right in [("train", "val"), ("train", "test"), ("val", "test")]:
    assert query_sets[left].isdisjoint(query_sets[right]), f"Query overlap: {left}/{right}"
assert items["example_id"].is_unique, "A source example ID was reused."
assert pointwise.groupby("query_id").size().max() <= 3
assert examples.loc[examples["format"].eq("listwise"), "product_count"].between(4, 8).all()
for records in records_by_split.values():
    for record in records:
        meta, answer = record["meta"], record["messages"][-1]["content"]
        assert set(meta["labels"]).issubset(LABELS)
        if meta["format"] == "pointwise":
            assert answer in LABELS and answer == meta["labels"][0]
        else:
            assert json.loads(answer) == {str(i): label for i, label in enumerate(meta["labels"], 1)}
print(f"Passed: disjoint query splits, {len(items):,} unique source examples, correct list sizes, valid answers, and training query cap.")

### Try these small investigations

1. Search for a product category you know. Read the user message and predict the label before reading the answer.
2. Find a Substitute and a Complement. What product attribute separates them?
3. Compare training pointwise label shares with validation/test. Which differences come from the deliberate training quotas?

Original annotations can be debatable. Note questionable cases rather than assuming every saved label is perfect. The readable JSON and this notebook are views of the prepared dataset; they do not change its labels.